In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from data_preprocessing.repartition import Repartition
from data_preprocessing.reconciliation import Reconciliation
from feature_engineering.order_flow import OrderFlowFeatureEngineering
from feature_engineering.order_trade_join import OrderTradeFeatureJoin

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

# Shared configuration
BAR_DURATION_MS = 250

In [ ]:
# Reconciliation Pipeline
config_recon = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://orderflowanalysis/intermediate/normalized',
        start_date='2024-12-31',
        end_date='2024-12-31',
        exchanges=['AMERICAS'],
        data_types=['trades', 'level2q']
    ),
    processing=ProcessingConfig(
        feature_engineering=OrderFlowFeatureEngineering(
            bar_duration_ms=BAR_DURATION_MS,
            max_section=2,
            group_filter=["file_count < 1 or file_count > 400"]
        )
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        repartitioned=S3Location(path='s3://orderflowanalysis/intermediate/repartitioned_v3'),
        reconciliation=S3Location(path='s3://orderflowanalysis/intermediate/reconciliation'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest'),
        metadata=S3Location(path='s3://orderflowanalysis/metadata')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir}, flat_core_count=17)
)

print("Testing Feature Engineering Pipeline...")
pipeline_recon = Pipeline(config_recon)

# Run all files
results_recon = pipeline_recon.run()  # Process all discovered files

# Or run with slice: results_recon = pipeline_recon.run(files_slice=slice(5))

In [ ]:
# Verify reconciliation results
if results_recon:
    res_pd = pd.DataFrame(results_recon)
    print(f"Total: {len(results_recon)}, Success: {(res_pd['message']=='success').sum()}, Failed: {(res_pd['message']!='success').sum()}")
    
    successful = [r for r in results_recon if r.get('message') == 'success']
    if successful:
        total_matched = sum(r['matched_groups'] for r in successful)
        total_mismatched = sum(r['mismatched_groups'] for r in successful)
        print(f"\nTotal matched groups: {total_matched:,}")
        print(f"Total mismatched groups: {total_mismatched:,}")
        
        print("\nSample results:")
        for r in successful[:10]:
            print(f"{r['date']} / {r['data_type']}: {r['matched_groups']:,} matched, {r['mismatched_groups']:,} mismatched")

In [ ]:
# Test Join Pipeline
config_join = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://orderflowanalysis/intermediate/features',
        start_date='2024-12-31',
        end_date='2024-12-31',
        exchanges=['AMERICAS'],
        data_types=['trades', 'level2q']
    ),
    processing=ProcessingConfig(
        join=OrderTradeFeatureJoin(
            bar_duration_ms=BAR_DURATION_MS,
            max_retries=3
        )
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        repartitioned=S3Location(path='s3://orderflowanalysis/intermediate/repartitioned_v3'),
        reconciliation=S3Location(path='s3://orderflowanalysis/intermediate/reconciliation'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest'),
        metadata=S3Location(path='s3://orderflowanalysis/metadata')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir}, flat_core_count=17)
)

print("Testing Join Pipeline...")
pipeline_join = Pipeline(config_join)

# Run join step
results_join = pipeline_join.run(files_slice=slice(5))  # Process first 5 file pairs

In [ ]:
# Verify join results
if results_join:
    res_pd = pd.DataFrame(results_join)
    print(f"Total: {len(results_join)}, Success: {(res_pd['message']=='success').sum()}, Failed: {(res_pd['message']!='success').sum()}")
    
    successful = [r for r in results_join if r.get('message') == 'success']
    if successful:
        print(f"\nSample join results:")
        for r in successful[:5]:
            print(f"{r['input_path']} -> {r['output_path']} ({r['row_count']:,} rows)")